## 📦 Requirements

Install Gradio and Plotly if not already present. Both are lightweight and don't affect
the training environment they're only needed for the demo interface.

In [1]:
%pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.9 MB 7.2 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/9.9 MB 7.7 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.9 MB 8.0 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.9 MB 7.9 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.9 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 8.3 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## ⚙️ Setup

Loads the tokenizer (same one used during training must match exactly) and defines
the color scheme used for the three classes throughout the interface.

In [3]:
import torch
import gradio as gr
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from transformers import AutoTokenizer
from model_utils.model_builder import HateSpeechClassifier
from dataset.df_loader import LABEL_MAP

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilroberta-base')

CLASS_COLORS = {
    'hatespeech': '#e74c3c',
    'offensive':  '#e67e22',
    'normal':     '#2ecc71',
}
print(f'Device: {DEVICE}')

Device: cuda


## 📂 Load models

Loads the four checkpoints saved at the end of training. Each model has the same architecture
(DistilRoBERTa + custom head) but different weights reflecting the different training strategies.
If a model file is missing, run the corresponding experiment cell in `main.ipynb` first.

In [4]:
def load_model(path):
    model = HateSpeechClassifier(num_labels=3)
    model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    return model.to(DEVICE).eval()

MODELS = {
    'Baseline':   load_model('models/Baseline.pt'),
    'Replay':     load_model('models/Replay.pt'),
    'Replay+EWC': load_model('models/Replay_EWC.pt'),
    'DER++':      load_model('models/DER__.pt'),
}
print(f'Loaded: {list(MODELS.keys())}')

Loaded: ['Baseline', 'Replay', 'Replay+EWC', 'DER++']


## 🚀 Launch demo

Builds the Gradio interface and launches it at `localhost:7860`. 
Type any sentence in the text box, select which models to compare, and click Classify.
Six example sentences are provided, including clear cases and ambiguous ones, to highlight
where the strategies agree and where they diverge.
The predicted class is shown in bold with its color; non-predicted classes are dimmed.

In [9]:
def predict(text, model):
    enc = tokenizer(text, return_tensors='pt', truncation=True,
                    max_length=128, padding=True).to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits[0]
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    label = list(LABEL_MAP.keys())[probs.argmax()]
    return label, dict(zip(LABEL_MAP.keys(), probs))


def plot_predictions(text, selected_models):
    if not text.strip() or not selected_models:
        return None

    selected = {name: MODELS[name] for name in selected_models if name in MODELS}
    classes  = list(LABEL_MAP.keys())
    n        = len(selected)

    fig = make_subplots(
        rows=1, cols=n,
        subplot_titles=list(selected.keys()),
        horizontal_spacing=0.08,
    )

    for col, (name, model) in enumerate(selected.items(), start=1):
        label, probs = predict(text, model)

        for cls in classes:
            val    = probs[cls]
            is_top = cls == label
            fig.add_trace(
                go.Bar(
                    x=[val],
                    y=[cls],
                    orientation='h',
                    marker=dict(
                        color=CLASS_COLORS[cls],
                        opacity=1.0 if is_top else 0.35,
                        line=dict(width=3 if is_top else 0,
                                  color=CLASS_COLORS[cls]),
                    ),
                    text=f'{val:.1%}',
                    textposition='outside',
                    textfont=dict(size=13, color=CLASS_COLORS[cls] if is_top else '#ccc'),
                    showlegend=False,
                    hovertemplate=f'<b>{cls}</b>: {val:.3f}<extra></extra>',
                ),
                row=1, col=col,
            )

        fig.layout.annotations[col - 1].update(
            text=f'<b>{name}</b><br><span style="color:{CLASS_COLORS[label]};font-size:13px">→ {label}</span>',
            font=dict(size=14, color='#ccc'),
        )

        fig.update_xaxes(range=[0, 1.18], showgrid=False,
                         zeroline=False, showticklabels=False,
                         row=1, col=col)
        fig.update_yaxes(tickfont=dict(size=13, color='#ccc'), row=1, col=col)

    display_text = text if len(text) <= 90 else text[:87] + '...'
    fig.update_layout(
        title=dict(text=f'<i>"{display_text}"</i>', font=dict(size=13, color='#ccc'), x=0.5),
        height=260,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=20, r=20, t=90, b=20),
        bargap=0.35,
    )
    return fig


EXAMPLES = [
    'I hate all people from that country, they should leave.',
    'You are such an idiot, nobody can stand you.',
    'The weather today is absolutely beautiful, I love it!',
    'Those people are ruining everything we worked for.',
    'Great job on the presentation, you did really well!',
    'Kill all of them, they do not deserve to live here.',
]

with gr.Blocks(title='Hate Speech Detection', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🔍 Hate Speech Detection — Continual Learning Demo
    Compare how different anti-forgetting strategies classify hate speech.
    Models were trained sequentially on **Davidson** → **HateXplain**.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label='Input Text',
                placeholder='Type a sentence to classify...',
                lines=3,
            )
            model_selector = gr.CheckboxGroup(
                choices=list(MODELS.keys()),
                value=list(MODELS.keys()),
                label='Models to compare',
            )
            submit_btn = gr.Button('Classify', variant='primary', size='lg')

        with gr.Column(scale=1):
            gr.Markdown("""
            ### Classes
            🔴 **hatespeech** — targeted hate toward a group  
            🟠 **offensive** — offensive but not hateful  
            🟢 **normal** — neutral content

            ### Strategies
            - **Baseline** — no anti-forgetting
            - **Replay** — replays past samples
            - **Replay+EWC** — replay + parameter protection
            - **DER++** — replay + output distillation
            """)

    output_plot = gr.Plot(label='Predictions')

    gr.Examples(examples=EXAMPLES, inputs=text_input, label='Example sentences')

    submit_btn.click(fn=plot_predictions, inputs=[text_input, model_selector], outputs=output_plot)
    text_input.submit(fn=plot_predictions, inputs=[text_input, model_selector], outputs=output_plot)

demo.launch(share=False)

C:\Users\Sean\AppData\Local\Temp\ipykernel_28944\3158296568.py:82: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title='Hate Speech Detection', theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
